In [ ]:
"""pattern_segmentation.ipynb

Automatically generated by Colab.

"""

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks


#!unzip dataset-patterns.zip -d /content

# Parameters
IMG_HEIGHT = 496
IMG_WIDTH = 696
IMG_CHANNELS = 3
DATASET_PATH = "dataset-patterns"  # folder with images
BATCH_SIZE = 4
EPOCHS = 20

# Loading images and creating binary masks
images = []
masks = []

for file in os.listdir(DATASET_PATH):
    if file.endswith(".jpg") or file.endswith(".png"):
        img_path = os.path.join(DATASET_PATH, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        images.append(img)

        # Binary mask: grayscale and threshold
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 250, 1, cv2.THRESH_BINARY_INV)  # white background - 0, object - 1
        mask = np.expand_dims(mask, axis=-1)
        masks.append(mask)

images = np.array(images, dtype=np.float32) / 255.0
masks = np.array(masks, dtype=np.float32)

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

#  Training and test split
X_train, X_val, y_train, y_val = train_test_split(images, masks, test_size=0.2, random_state=42)

# U-Net model
def unet_model(input_size=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)):
    inputs = layers.Input(input_size)

    # Encoder
    c1 = layers.Conv2D(16, (3,3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(16, (3,3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2,2))(c1)

    c2 = layers.Conv2D(32, (3,3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(32, (3,3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2,2))(c2)

    c3 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2,2))(c3)

    # Bottleneck
    c4 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c4)

    # Decoder
    u5 = layers.UpSampling2D((2,2))(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c5)

    u6 = layers.UpSampling2D((2,2))(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = layers.Conv2D(32, (3,3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(32, (3,3), activation='relu', padding='same')(c6)

    u7 = layers.UpSampling2D((2,2))(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = layers.Conv2D(16, (3,3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(16, (3,3), activation='relu', padding='same')(c7)

    outputs = layers.Conv2D(1, (1,1), activation='sigmoid')(c7)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

model = unet_model()
model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

#
earlystopper = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS,
                    callbacks=[earlystopper])

# Prediction on validation set and displaying results
preds_val = model.predict(X_val)
preds_val_thresholded = (preds_val > 0.5).astype(np.uint8)

n = 5
plt.figure(figsize=(12,8))
for i in range(n):
    plt.subplot(n,3,i*3+1)
    plt.imshow(X_val[i])
    plt.title("Original")
    plt.axis('off')

    plt.subplot(n,3,i*3+2)
    plt.imshow(y_val[i].squeeze(), cmap='gray')
    plt.title("Mask")
    plt.axis('off')

    plt.subplot(n,3,i*3+3)
    plt.imshow(preds_val_thresholded[i].squeeze(), cmap='gray')
    plt.title("Prediction")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Analysis: Loss and accuracy during epochs
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.title('Loss during epochs')
plt.xlabel('Epocha')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train acc')
plt.plot(history.history['val_accuracy'], label='Val acc')
plt.title('Accuracy during epochs')
plt.xlabel('Epocha')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

# Analysis: Calculation and display of IoU (Intersection over Union)
def iou_score(y_true, y_pred):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    if union == 0:
        return 1.0  # If there is no pixel, IoU is 1
    return intersection / union

# IoU for every image in the validation set
ious = []
for i in range(len(y_val)):
    iou = iou_score(y_val[i], preds_val_thresholded[i])
    ious.append(iou)

print(f"Average IoU on validation set: {np.mean(ious):.4f}")

# Display histogram of IoU values
plt.figure(figsize=(6,4))
plt.hist(ious, bins=10, color='skyblue', edgecolor='black')
plt.title('Histogram IoU on validation set')
plt.xlabel('IoU')
plt.ylabel('Number of images')
plt.show()